# Week 2 — EfficientNet-B3 EF Regression
## Goal:
Train a pretrained EfficientNet-B3 model to predict
ejection fraction (EF) from echocardiography frames.

In [1]:
import cv2
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
from torchvision import models



In [2]:
ECHONET_DIR = Path("../../data/raw/EchoNet-Dynamic")
filelist = pd.read_csv(ECHONET_DIR/"FileList.csv")
tracings = pd.read_csv(ECHONET_DIR/"VolumeTracings.csv")

In [3]:
# Image preprocessing pipeline applied to every extracted frame
echo_tfm = transforms.Compose([
    # Convert OpenCV NumPy image into PIL format
    # because torchvision transforms work efficiently with PIL images
    transforms.ToPILImage(),
    # Resize all cardiac frames to a fixed resolution
    # so every input tensor has the same shape for CNN models
    transforms.Resize((224, 224)),
    # Convert image into PyTorch tensor
    # Shape changes from (H,W,C) → (C,H,W)
    # Pixel values also scale from [0,255] → [0,1]
    transforms.ToTensor(),
    # Normalize tensor using ImageNet statistics
    # Helps pretrained CNN models generalize better
    # and stabilizes training
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [4]:
def ef_to_severity(ef):
    if ef >= 50:
        return 0   # Normal
    elif ef >= 40:
        return 1   # Mild
    else:
        return 2   # Severe

In [5]:
def get_annotated_frames(video_name,tracings):
    video_tracings = tracings[tracings["FileName"] ==video_name]
    frames = sorted(video_tracings["Frame"].unique())

    if len(frames)>2:
        raise ValueError(f"{video_name} has {len(frames)} annotated frames: {frames}")

    return frames

In [6]:
def read_two_frames(video_path, frame_indices):
     # Open the echo video using OpenCV
    # video_path is converted to string because OpenCV expects string paths
    cap = cv2.VideoCapture(str(video_path))
    # Store transformed ED and ES frame tensors
    frames = []

    for idx in frame_indices:
         # Jump directly to the required frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()

        if not ok:
            cap.release()
            raise RuntimeError(f"Could not read frame {idx} from {video_path}")

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(echo_tfm(frame_rgb))

    cap.release()

    return torch.stack(frames)

In [7]:
def get_ed_es_frames_by_tracing_count(video_name, tracings):
     # Get annotated cardiac frames for the given video
    # EchoNet usually provides two annotated phases:
    # ED (expanded ventricle) and ES (contracted ventricle)
    frames = get_annotated_frames(video_name, tracings)

    lengths = {}

    for f in frames:
        frame_points = tracings[
            (tracings["FileName"] == video_name) &
            (tracings["Frame"] == f)
        ]

        # approximate ventricle size using average line length
        # Compute length of every tracing line segment
        # using Euclidean distance formula
        line_lengths = np.sqrt(
            (frame_points["X2"] - frame_points["X1"])**2 +
            (frame_points["Y2"] - frame_points["Y1"])**2
        )
        # Total tracing length is used as an approximate
        # estimate of ventricle size

        lengths[f] = line_lengths.sum()

    ed_frame = max(lengths, key=lengths.get)
    es_frame = min(lengths, key=lengths.get)

    return ed_frame, es_frame

In [8]:
def draw_contour(frame_rgb, video_name, frame_idx, tracings):

    frame_points = tracings[
        (tracings["FileName"] == video_name) &
        (tracings["Frame"] == frame_idx)
    ]

    plt.imshow(frame_rgb)

    for _, row in frame_points.iterrows():

        plt.plot(
            [row["X1"], row["X2"]],
            [row["Y1"], row["Y2"]],
            color="red"
        )

    plt.axis("off")

In [9]:
# Custom PyTorch Dataset for EchoNet-Dynamic preprocessing pipeline
class EchoDataset(Dataset):

    def __init__(self, filelist, tracings, root_dir):

        self.filelist = filelist
        self.tracings = tracings
        self.root_dir = root_dir

    def __len__(self):

        return len(self.filelist)

    def __getitem__(self, idx):

        row = self.filelist.iloc[idx]

        video_name = row["FileName"] + ".avi"

        video_path = self.root_dir / "Videos" / video_name

        ef = row["EF"]

        severity = ef_to_severity(ef)

        ed_frame, es_frame = get_ed_es_frames_by_tracing_count(
            video_name,
            self.tracings
        )

        tensor = read_two_frames(
            video_path,
            [ed_frame]
        )[0]

        return {
            "video_name": video_name,
            "tensor": tensor,
            "ef": ef,
            "severity": severity
        }

In [10]:
tracing_videos = set(tracings["FileName"].unique())

filelist["VideoNameWithExt"] = filelist["FileName"] + ".avi"

filelist_clean = filelist[
    filelist["VideoNameWithExt"].isin(tracing_videos)
].reset_index(drop=True)

print("Original filelist:", len(filelist))
print("Clean filelist:", len(filelist_clean))

Original filelist: 10030
Clean filelist: 10024


In [11]:
dataset = EchoDataset(
    filelist=filelist_clean,
    tracings=tracings,
    root_dir=ECHONET_DIR
)

train_loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

In [12]:
sample = dataset[0]

print(sample["video_name"])

print(sample["tensor"].shape)

print(sample["ef"])

0X100009310A3BD7FC.avi
torch.Size([3, 224, 224])
78.49840597


# EfficientNet-B3 Encoder
## Transfer Learning for EF Regression

In [13]:
model = models.efficientnet_b3(
    weights="IMAGENET1K_V1"
)

In [14]:
print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=40, bias=False)
            (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(40, 10, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(10, 40, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
        

In [15]:
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(1536, 512),
    nn.ReLU(),
    nn.Linear(512, 1)
)

In [16]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1536, out_features=512, bias=True)
  (2): ReLU()
  (3): Linear(in_features=512, out_features=1, bias=True)
)


In [17]:
criterion = nn.L1Loss()

In [18]:
train_loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

In [20]:
criterion = nn.L1Loss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [21]:
batch = next(iter(train_loader))

images = batch["tensor"].to(device)
efs = batch["ef"].float().view(-1, 1).to(device)

print(images.shape)
print(efs.shape)

torch.Size([8, 3, 224, 224])
torch.Size([8, 1])


In [22]:
preds = model(images)

print(preds.shape)

torch.Size([8, 1])


In [23]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [24]:
model.train()

preds = model(images)

loss = criterion(preds, efs)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 3451.87744140625


In [ ]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        images = batch["tensor"].to(device)

        efs = batch["ef"].float().view(-1,1).to(device)

        preds = model(images)

        loss = criterion(preds, efs)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")